# ME2 - VCM on Raspberry Pi 5 — Personalized Wake and Intent Retraining

This experiment uses the locally recorded `person-01` audio. It compares two independently scratch-initialized two-class wake detectors:

1. **Broad negatives:** real wake takes as positives; Option B command speech, silence and background noise as non-wake.
2. **Personal hard negatives:** the same wake takes and broad negatives, plus the speaker's own non-wake commands and near-miss speech.

Training on wake recordings alone cannot teach a wake/non-wake decision; a binary detector needs negative examples. The intent candidate is scratch-trained on the Option B training partition plus a small, controlled fraction of exact-mapped personal command recordings. The canonical Option B INT8 model is the intent baseline. All splits are made after exact PCM deduplication. The Option B held-out test data are not loaded until the final cell, after all models and thresholds are frozen.

**Important limits:** the personal recordings are from one speaker and one day, so their held-out results measure only this speaker's recorded clips. Several fixed intent classes have very few unique personal takes. The Option B test split has been used in prior project experiments. No Pi performance claim is made by these notebook results.

Normal execution trains from scratch and evaluates once. Set `VCM_EXISTING_RUN_DIR` before executing to reconstruct the visible report from an already completed run without retraining or reopening the held-out Option B test.

In [1]:
from pathlib import Path
import os, sys, json, time
from datetime import datetime
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'tinyvcm_antigrav').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from its ME2 project folder so tinyvcm_antigrav can be found.')
WORKSPACE_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from tinyvcm_antigrav.personalized_retrain import (
    prepare_experiment, run_wake_pair, train_intent_candidate_from_data,
    _export_intent_candidate, evaluate_all, INTENT_MAP,
    load_personal_records, load_optionb_train_val,
)

REUSE_RUN_DIR = Path(os.environ['VCM_EXISTING_RUN_DIR']).resolve() if os.environ.get('VCM_EXISTING_RUN_DIR') else None
RUN_DIR = REUSE_RUN_DIR or (PROJECT_ROOT / 'runs' / f"personalized-vcm-{datetime.now().strftime('%Y%m%d-%H%M%S')}")
print('Project:', PROJECT_ROOT)
print('Existing completed run for report reconstruction:' if REUSE_RUN_DIR else 'New scratch training output:', RUN_DIR)
print('Shared Python:', sys.executable)


Project: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\AI 231\Dumalaog_ME2 - Tiny Voice Command Model
Existing completed run for report reconstruction: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\AI 231\Dumalaog_ME2 - Tiny Voice Command Model\runs\personalized-vcm-20260928-155109
Shared Python: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\.venv\Scripts\python.exe


## 1. Audit, deduplicate, split, and prepare training data

This cell validates the local manifest and WAV headers, removes exact duplicate PCM before splitting, writes a split manifest under the new run, and reads only the Option B training and validation feature arrays. It deliberately does not open the Option B test features.

In [2]:
if REUSE_RUN_DIR:
    human, _ = load_personal_records(PROJECT_ROOT)
    audit = json.loads((RUN_DIR / 'dataset_audit.json').read_text(encoding='utf-8'))
    DATA = {**load_optionb_train_val(PROJECT_ROOT), 'project_root': PROJECT_ROOT,
            'human': human, 'human_audit': audit, 'output_dir': RUN_DIR}
    print('Reconstructed the deterministic personal data split. Train/validation features only; test is not opened.')
else:
    DATA = prepare_experiment(RUN_DIR, PROJECT_ROOT)
audit = DATA['human_audit']
display(pd.DataFrame([
    {'saved files': audit['raw_manifest_rows'], 'unique waveforms': audit['unique_pcm_groups'],
     'duplicate rows removed': audit['exact_duplicate_rows_collapsed'],
     'real speakers': ', '.join(audit['speakers']), 'cross-label PCM collisions': len(audit['cross_label_hash_collisions'])}
]))
display(pd.DataFrame([
    {'label': label, 'unique takes': count,
     'train': sum(i['label']==label and i['split']=='train' for i in DATA['human']),
     'validation': sum(i['label']==label and i['split']=='validation' for i in DATA['human']),
     'test': sum(i['label']==label and i['split']=='test' for i in DATA['human']),
     'mapped intent': INTENT_MAP.get(label, 'wake / not mapped')}
    for label, count in audit['unique_label_groups'].items()
]))
print('Option B cache manifest SHA-256:', DATA['manifest_sha256'].item())
print('Personal manifest SHA-256:', audit['manifest_sha256'])


Reconstructed the deterministic personal data split. Train/validation features only; test is not opened.


,saved files,unique waveforms,duplicate rows removed,real speakers,cross-label PCM collisions
0,428,201,227,person-01,0


,label,unique takes,train,validation,test,mapped intent
0,_background_noise_,5,3,1,1,wake / not mapped
1,_silence_,3,1,1,1,wake / not mapped
2,_unknown_,4,2,1,1,wake / not mapped
3,alarm_set,11,7,1,3,wake / not mapped
4,call_mom,7,4,1,2,CALL
5,lights_off,24,16,3,5,LIGHT_OFF
6,lights_on,7,4,1,2,LIGHT_ON
7,media_next,8,5,1,2,NEXT
8,media_pause,7,4,1,2,PAUSE
9,media_resume,6,4,1,1,wake / not mapped


Option B cache manifest SHA-256: 3a7cb34b708414df055bb16a15ed2c9f7d850f450c88f02602079e06e961ab23
Personal manifest SHA-256: c2625ad238f0206bd0276efd62c346bb21f736034e7ba6b6094b6a4025a1b10d


## 2. Train the wake-model comparison

Each variant sees the same speaker-held-out wake clips, Option B command pool, and training schedule. The second variant adds only training-split personal command/near-miss recordings. Validation selects each INT8 threshold under a zero-validation-false-accept constraint. The final test clips remain unopened.

In [3]:
if REUSE_RUN_DIR:
    WAKE_RUNS = {}
    for name, folder in (('broad', 'wake_broad_negatives'), ('hard', 'wake_with_personal_intents')):
        model_dir = RUN_DIR / folder / 'models'
        summary = json.loads((model_dir / 'export_summary.json').read_text(encoding='utf-8'))
        WAKE_RUNS[name] = {
            'summary': summary,
            'val_threshold': summary['validation_threshold_int8'],
            'fp32_path': model_dir / 'wake_personalized_fp32.onnx',
            'int8_path': model_dir / 'wake_personalized_int8.onnx',
        }
        history = json.loads((RUN_DIR / folder / 'history.json').read_text(encoding='utf-8'))['history']
        print(f"Loaded {name} wake model trained for {len(history)} epochs; selected epoch {summary['best_epoch']}.")
        display(pd.DataFrame(history))
else:
    WAKE_RUNS = run_wake_pair(DATA, epochs=30)
display(pd.DataFrame([
    {'variant': name, 'INT8 bytes': run['summary']['int8_size_bytes'],
     'validation threshold': run['val_threshold'],
     'FP32 validation wake recall': run['summary']['fp32_validation_threshold_metrics']['wake_recall'],
     'validation wake recall': run['summary']['int8_validation_threshold_metrics']['wake_recall'],
     'validation wake support': run['summary']['int8_validation_threshold_metrics']['wake_support'],
     'validation non-wake support': run['summary']['int8_validation_threshold_metrics']['nonwake_support'],
     'INT8 SHA-256': run['summary']['int8_sha256']}
    for name, run in WAKE_RUNS.items()
]))


Loaded broad wake model trained for 30 epochs; selected epoch 27.


,epoch,train_loss,train_accuracy,val_zero_FA_wake_recall,val_threshold,val_wake_support,val_nonwake_clip_support,epoch_seconds
0,1,0.268101,0.952624,0.000000,1.000000,3,1842,3.211411
1,2,0.097066,0.984329,0.000000,1.000000,3,1842,1.962254
2,3,0.052349,0.991254,0.666667,0.786128,3,1842,2.040653
3,4,0.026484,0.997813,0.666667,0.954816,3,1842,2.047424
4,5,0.010810,1.000000,0.666667,0.853077,3,1842,1.964188
5,6,0.009969,0.998542,1.000000,0.880256,3,1842,1.982341
6,7,0.010178,0.999271,1.000000,0.734193,3,1842,2.005636
7,8,0.006847,0.999271,0.666667,0.941375,3,1842,1.943984
8,9,0.004911,1.000000,1.000000,0.942229,3,1842,1.961796
9,10,0.004629,0.999636,1.000000,0.991336,3,1842,1.953117


Loaded hard wake model trained for 30 epochs; selected epoch 17.


,epoch,train_loss,train_accuracy,val_zero_FA_wake_recall,val_threshold,val_wake_support,val_nonwake_clip_support,epoch_seconds
0,1,0.280622,0.952999,0.000000,1.000000,3,1842,2.225099
1,2,0.135215,0.967029,0.000000,1.000000,3,1842,2.001664
2,3,0.089359,0.974746,0.666667,0.859965,3,1842,2.055260
3,4,0.054195,0.989477,1.000000,0.968645,3,1842,2.031060
4,5,0.029894,0.993686,1.000000,0.334366,3,1842,2.036219
5,6,0.020231,0.997545,1.000000,0.403911,3,1842,2.076730
6,7,0.016233,0.996492,1.000000,0.079984,3,1842,2.029518
7,8,0.012307,0.998246,1.000000,0.977501,3,1842,2.025045
8,9,0.012061,0.998597,1.000000,0.940260,3,1842,2.035882
9,10,0.011059,0.998597,1.000000,0.934771,3,1842,2.026963


,variant,INT8 bytes,validation threshold,FP32 validation wake recall,validation wake recall,validation wake support,validation non-wake support,INT8 SHA-256
0,broad,37425,0.999218,1.0,1.0,3,1842,5e74fade5beefdd86267823ea0c895fcd3a512fc04ce87...
1,hard,37425,0.999942,1.0,1.0,3,1842,83d6f765133b42b34c9a6eed426e9ffd98113bb4f3979d...


## 3. Train the personalized intent candidate from scratch

This retains the 31 fixed Option B output classes. Only exact-mapped personal labels are added; unsupported phrases such as five-minute timers, thermostat direction, and the recorded seven-AM alarm are excluded instead of being mislabeled. The per-batch personal fraction is intentionally small. Checkpoint selection uses validation only.

In [4]:
if REUSE_RUN_DIR:
    INTENT_DIR = RUN_DIR / 'intent_personalized'
    INTENT_RUN = {'output_dir': INTENT_DIR,
                  'run_info': json.loads((INTENT_DIR / 'run_info.json').read_text(encoding='utf-8'))}
    INTENT_EXPORT = json.loads((INTENT_DIR / 'models' / 'export_summary.json').read_text(encoding='utf-8'))
    history = json.loads((INTENT_DIR / 'history.json').read_text(encoding='utf-8'))['history']
    print(f"Loaded scratch-trained personalized intent model; selected epoch {INTENT_RUN['run_info']['best_epoch']}.")
    display(pd.DataFrame(history))
else:
    INTENT_RUN = train_intent_candidate_from_data(DATA, epochs=35)
    INTENT_EXPORT = _export_intent_candidate(INTENT_RUN, DATA)
display(pd.DataFrame([{
    'epochs': INTENT_RUN['run_info']['epochs'], 'best epoch': INTENT_RUN['run_info']['best_epoch'],
    'personal train clips': INTENT_RUN['run_info']['personal_unique_train_clips'],
    'parameter count': INTENT_RUN['run_info']['parameter_count'],
    'FP32 bytes': INTENT_EXPORT['fp32_size_bytes'], 'INT8 bytes': INTENT_EXPORT['int8_size_bytes'],
    'INT8 SHA-256': INTENT_EXPORT['int8_sha256']
}]))


Loaded scratch-trained personalized intent model; selected epoch 34.


,epoch,train_loss,train_accuracy,val_optionb_macro_f1,val_personal_macro_f1,checkpoint_score,epoch_seconds
0,1,2.829581,0.143333,0.157197,0.079365,0.141630,10.491103
1,2,2.204568,0.291042,0.271550,0.098485,0.236937,9.958607
2,3,1.812240,0.446042,0.474654,0.192460,0.418215,9.955019
3,4,1.528353,0.565139,0.544819,0.413889,0.518633,9.972502
4,5,1.314479,0.648681,0.716812,0.583333,0.690116,10.247308
5,6,1.168116,0.700486,0.758441,0.541667,0.715086,9.950604
6,7,1.069655,0.741875,0.725628,0.273810,0.635265,9.950339
7,8,0.991031,0.767917,0.837092,0.736111,0.816896,9.979803
8,9,0.936980,0.792222,0.843660,0.543651,0.783658,9.945119
9,10,0.884922,0.814514,0.874365,0.677778,0.835048,9.951896


,epochs,best epoch,personal train clips,parameter count,FP32 bytes,INT8 bytes,INT8 SHA-256
0,35,34,73,14527,60466,39315,c91c73860b24eea13b74620970febd18c1cf531629de72...


## 4. One-time held-out evaluation

Only now are the Option B test features opened. The threshold and selected checkpoints are frozen. Personal results are at the unique-waveform level; repeated recorder saves cannot multiply the test support. During report reconstruction, this cell reads the saved one-time evaluation and does not re-open the test set.

In [5]:
if REUSE_RUN_DIR:
    RESULTS = json.loads((RUN_DIR / 'final_evaluation.json').read_text(encoding='utf-8'))
    print('Loaded the saved final evaluation; held-out Option B data were not evaluated again.')
else:
    RESULTS = evaluate_all(DATA, WAKE_RUNS, INTENT_RUN)
print('Wake comparison:')
display(pd.DataFrame([
    {'variant': name, 'threshold': r['validation_selected_int8_threshold'],
     'FP32 wake hits/support': f"{r['fp32_heldout_personal_wake_hits']}/{r['fp32_heldout_personal_wake_support']}",
     'FP32 wake recall': r['fp32_heldout_personal_wake_recall'],
     'personal wake recall': r['heldout_personal_wake_recall'],
     'personal wake hits/support': f"{r['heldout_personal_wake_hits']}/{r['heldout_personal_wake_support']}",
     'personal non-wake false accepts': f"{r['false_wake_on_personal_nonwake_clips']}/{r['personal_nonwake_support']}",
     'Option B command false accepts': f"{r['false_wake_on_optionb_command_clips']}/{r['optionb_command_support']}",
     'model bytes': r['int8_model_size_bytes'], 'PC CPU p95 ms': r['pc_cpu_latency']['p95_ms_pc_cpu_model_only']}
    for name, r in RESULTS['wake_model_comparison'].items()
]))
print('Intent comparison:')
display(pd.DataFrame([
    {'model': name,
     'Option B INT8/test accuracy': vals['optionb_test']['accuracy'],
     'Option B macro-F1': vals['optionb_test']['macro_f1_supported_classes'],
     'personal heldout count': vals['personal_heldout_test'].get('count', 0),
     'personal heldout accuracy': vals['personal_heldout_test'].get('accuracy'),
     'personal macro-F1': vals['personal_heldout_test'].get('macro_f1_supported_classes'),
     'size bytes': vals['model_size_bytes'], 'PC CPU p95 ms': vals['pc_cpu_latency']['p95_ms_pc_cpu_model_only']}
    for name, vals in RESULTS['intent_model_comparison'].items()
]))
print('All 31 intent classes on the held-out Option B speakers (canonical vs personalized INT8):')
CLASSIC = RESULTS['intent_model_comparison']['canonical_int8']['optionb_test']['classification_report']
PERSONAL = RESULTS['intent_model_comparison']['personalized_int8']['optionb_test']['classification_report']
display(pd.DataFrame([
    {'class': label, 'support': int(CLASSIC[label]['support']),
     'canonical recall': CLASSIC[label]['recall'], 'personalized recall': PERSONAL[label]['recall'],
     'canonical F1': CLASSIC[label]['f1-score'], 'personalized F1': PERSONAL[label]['f1-score']}
    for label in CLASSIC if label in PERSONAL and isinstance(CLASSIC[label], dict) and 'f1-score' in CLASSIC[label]
]))
print('Your held-out recordings, grouped by recorder label (unmapped labels are prediction-only):')
display(pd.DataFrame([
    {'model': model_name, 'recorded label': source_label,
     'held-out clips': row['heldout_clip_support'], 'mapped intent': row['mapped_intent'],
     'correct': row['correct_hits'], 'accuracy': row['accuracy'],
     'predicted classes': row['predicted_class_counts']}
    for model_name in ('canonical_int8', 'personalized_int8')
    for source_label, row in RESULTS['intent_model_comparison'][model_name]['personal_recorded_commands_all_heldout']['per_recorded_label'].items()
]))
print('Full report:', RUN_DIR / 'final_evaluation.json')
print(json.dumps(RESULTS['known_limitations'], indent=2))


Loaded the saved final evaluation; held-out Option B data were not evaluated again.
Wake comparison:


,variant,threshold,FP32 wake hits/support,FP32 wake recall,personal wake recall,personal wake hits/support,personal non-wake false accepts,Option B command false accepts,model bytes,PC CPU p95 ms
0,broad,0.999218,5/5,1.0,0.8,4/5,0/42,0/1798,37425,0.516295
1,hard,0.999942,4/5,0.8,0.8,4/5,0/42,0/1798,37425,0.510515


Intent comparison:


,model,Option B INT8/test accuracy,Option B macro-F1,personal heldout count,personal heldout accuracy,personal macro-F1,size bytes,PC CPU p95 ms
0,canonical_fp32,0.961068,0.960818,28,0.428571,0.353704,60466,0.402070
1,canonical_int8,0.962736,0.962536,28,0.428571,0.353704,39315,0.499615
2,personalized_fp32,0.962180,0.962017,28,0.857143,0.860317,60466,0.427440
3,personalized_int8,0.961068,0.960918,28,0.857143,0.860317,39315,0.492500


All 31 intent classes on the held-out Option B speakers (canonical vs personalized INT8):


,class,support,canonical recall,personalized recall,canonical F1,personalized F1
0,ALARM_6_00AM,58,0.982759,0.965517,0.974359,0.957265
1,ALARM_8_00AM,56,0.946429,0.982143,0.963636,0.948276
2,ALARM_9_00PM,58,1.000000,0.965517,0.974790,0.973913
3,BRIGHTNESS_100,60,0.983333,0.983333,0.951613,0.914729
4,BRIGHTNESS_20,60,0.983333,0.933333,0.975207,0.957265
5,BRIGHTNESS_60,60,1.000000,1.000000,1.000000,1.000000
6,CALL,54,0.944444,1.000000,0.935780,0.964286
7,COLOR_BLUE,60,0.966667,0.933333,0.950820,0.957265
8,COLOR_GREEN,60,0.933333,1.000000,0.965517,1.000000
9,COLOR_RED,60,1.000000,0.983333,1.000000,0.983333


Your held-out recordings, grouped by recorder label (unmapped labels are prediction-only):


,model,recorded label,held-out clips,mapped intent,correct,accuracy,predicted classes
0,canonical_int8,alarm_set,3,NaN,NaN,NaN,{'ALARM_8_00AM': 3}
1,canonical_int8,call_mom,2,CALL,0.0,0.000000,"{'LIGHT_ON': 1, 'VOLUME_UP': 1}"
2,canonical_int8,lights_off,5,LIGHT_OFF,0.0,0.000000,"{'ALARM_8_00AM': 1, 'LIGHT_ON': 4}"
3,canonical_int8,lights_on,2,LIGHT_ON,2.0,1.000000,{'LIGHT_ON': 2}
4,canonical_int8,media_next,2,NEXT,0.0,0.000000,{'WEATHER': 2}
5,canonical_int8,media_pause,2,PAUSE,0.0,0.000000,{'PLAY_MUSIC': 2}
6,canonical_int8,media_resume,1,NaN,NaN,NaN,{'PLAY_MUSIC': 1}
7,canonical_int8,play_music,1,PLAY_MUSIC,1.0,1.000000,{'PLAY_MUSIC': 1}
8,canonical_int8,question_time,3,TIME,2.0,0.666667,"{'TIME': 2, 'LIST_REMINDERS': 1}"
9,canonical_int8,question_weather,2,WEATHER,2.0,1.000000,{'WEATHER': 2}


Full report: C:\Users\danda\Desktop\MEng AI Notebooks\AI 222 231\AI 231\Dumalaog_ME2 - Tiny Voice Command Model\runs\personalized-vcm-20260928-155109\final_evaluation.json
[
  "Only one real personal speaker is present; accuracy is within-speaker clip holdout, not unseen-speaker accuracy.",
  "Personal held-out support is low for several intent classes and is reported per class.",
  "Option B test clips have prior project evaluations and are not a pristine first-use benchmark.",
  "Command-window false wake counts are not continuous-audio false activations per hour; no Pi hardware was used.",
  "INT8 latency here is local Windows PC CPU model-only latency, not Raspberry Pi frontend-plus-model latency.",
  "This experiment does not automatically replace the canonical model files or change the Pi release."
]


## Interpretation and promotion gate

Choose the wake variant using the shared held-out personal wake and non-wake results, while remembering there is only one real speaker. Keep the simpler broad-negative version if personal hard negatives do not improve wake recall without added false accepts. Keep the canonical intent model if the personalized model's personal gain damages the Option B speaker-disjoint score. These are local research candidates; exporting ONNX does not promote them to the UI or Raspberry Pi. Collect more speakers, sessions, wake near-misses, and room noise before treating them as general-purpose.